# SI3008 — Grupo 4
## Del modelo entidad-relación al esquema estrella en un lago de datos

**Rol A — Modelador/a de datos.** Este notebook construye el esquema estrella, lo materializa en Parquet (plano y particionado) y CSV, y valida la integridad referencial.

> Si tienes el archivo oficial `yellow_tripdata_2026-01.parquet`, colócalo en `datos/`. Si no, ejecuta primero la celda del generador sintético (semilla 2026).

### 0. Dependencias

In [ ]:
# !pip install pandas pyarrow matplotlib numpy
import os, numpy as np, pandas as pd
os.makedirs('datos', exist_ok=True)

### 1. (Opcional) Generar datos sintéticos si no tienes el archivo oficial

In [ ]:
"""
SI3008 - Grupo 4
Script 0: Generación de un dataset SINTÉTICO de taxis amarillos NYC (enero 2026)
que reproduce las propiedades verificadas del archivo oficial.

IMPORTANTE PARA EL/LA ESTUDIANTE:
-----------------------------------
La docente entrega el archivo real `yellow_tripdata_2026-01.parquet`.
Si ya lo tienes, OMITE este script y usa el archivo real.
Este generador sirve para que el notebook sea ejecutable de principio a fin
aunque todavía no tengas el archivo oficial. Documenta la semilla (SEED=2026)
en la bitácora si usas datos sintéticos.

Propiedades que reproduce (Anexo A del enunciado):
- ~3.72M registros, 20 columnas (aquí usamos una muestra reducida por memoria)
- columna cbd_congestion_fee con valores {0.75, 0.00, -0.75}
- patrón de nulos ligado a payment_type == 0 (~29.2%)
- algunos fare_amount <= 0, trip_distance <= 0, trip_distance > 100
- rango de fechas que desborda levemente el mes
"""
import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)

# Muestra reducida para que corra en Colab/entornos con poca RAM.
# Para el trabajo real usa el archivo completo de la docente.
N = 200_000

# --- Zonas (PULocationID / DOLocationID) 1..263 como en la TLC ---
zones = np.arange(1, 264)
pu = rng.choice(zones, size=N)
do = rng.choice(zones, size=N)

# --- Fechas dentro de enero 2026, con un pequeño desborde a dic-2025 y feb-2026 ---
base = pd.Timestamp("2026-01-01")
offsets_min = rng.integers(-180, 44700, size=N)  # minutos; algunos negativos => dic 2025
pickup = (base + pd.to_timedelta(offsets_min, unit="m")).to_numpy()
duration_min = rng.gamma(shape=2.0, scale=7.0, size=N) + 1  # duración viaje
dropoff = pickup + pd.to_timedelta(duration_min, unit="m").to_numpy()

# Inyectar 1 registro con dropoff < pickup (como el hallazgo oficial)
dropoff[0] = pickup[0] - pd.Timedelta(minutes=5).to_timedelta64()

# --- Medidas del viaje ---
trip_distance = np.round(rng.gamma(shape=2.0, scale=1.8, size=N), 2)
fare_amount = np.round(3.0 + trip_distance * rng.uniform(2.5, 3.5, size=N), 2)

# Inyectar anomalías
neg_fare_idx = rng.choice(N, size=int(N*0.011), replace=False)   # ~1.1% fare<=0
fare_amount[neg_fare_idx] = -np.abs(fare_amount[neg_fare_idx])
zero_dist_idx = rng.choice(N, size=int(N*0.034), replace=False)  # ~3.4% dist<=0
trip_distance[zero_dist_idx] = 0.0
far_idx = rng.choice(N, size=int(N*0.00004)+1, replace=False)    # muy pocos >100 millas
trip_distance[far_idx] = rng.uniform(101, 150, size=len(far_idx))

# --- payment_type: 0 marca el subgrupo con nulos (~29.2%) ---
payment_type = rng.choice([1, 2, 3, 4], size=N, p=[0.68, 0.27, 0.03, 0.02]).astype(float)
null_mask = rng.random(N) < 0.292
payment_type[null_mask] = 0.0

# Columnas que se vuelven nulas cuando payment_type == 0
passenger_count = rng.integers(1, 6, size=N).astype(float)
RatecodeID = rng.choice([1, 2, 3, 4, 5], size=N, p=[0.9, 0.04, 0.03, 0.02, 0.01]).astype(float)
store_and_fwd_flag = np.where(rng.random(N) < 0.02, "Y", "N").astype(object)
congestion_surcharge = np.round(rng.choice([0.0, 2.5], size=N, p=[0.2, 0.8]), 2)
Airport_fee = np.round(rng.choice([0.0, 1.75], size=N, p=[0.9, 0.1]), 2)

for col in [passenger_count, RatecodeID, congestion_surcharge, Airport_fee]:
    col[null_mask] = np.nan
store_and_fwd_flag = np.where(null_mask, None, store_and_fwd_flag)

# --- cbd_congestion_fee: {0.75, 0.00, -0.75} con proporciones del Anexo A ---
cbd = rng.choice([0.75, 0.00, -0.75], size=N, p=[0.70, 0.293, 0.007])

# --- Otras columnas monetarias para completar 20 columnas ---
VendorID = rng.choice([1, 2], size=N).astype("int64")
extra = np.round(rng.choice([0.0, 0.5, 1.0], size=N), 2)
mta_tax = np.full(N, 0.5)
tip_amount = np.where(payment_type == 1,
                      np.round(fare_amount * rng.uniform(0.0, 0.3, size=N), 2), 0.0)
tolls_amount = np.round(rng.choice([0.0, 6.55], size=N, p=[0.95, 0.05]), 2)
improvement_surcharge = np.full(N, 1.0)
total_amount = np.round(
    np.nan_to_num(fare_amount) + extra + mta_tax + tip_amount + tolls_amount +
    improvement_surcharge + np.nan_to_num(congestion_surcharge) +
    np.nan_to_num(Airport_fee) + cbd, 2)

df = pd.DataFrame({
    "VendorID": VendorID,
    "tpep_pickup_datetime": pickup,
    "tpep_dropoff_datetime": dropoff,
    "passenger_count": passenger_count,
    "trip_distance": trip_distance,
    "RatecodeID": RatecodeID,
    "store_and_fwd_flag": store_and_fwd_flag,
    "PULocationID": pu.astype("int64"),
    "DOLocationID": do.astype("int64"),
    "payment_type": payment_type,
    "fare_amount": fare_amount,
    "extra": extra,
    "mta_tax": mta_tax,
    "tip_amount": tip_amount,
    "tolls_amount": tolls_amount,
    "improvement_surcharge": improvement_surcharge,
    "total_amount": total_amount,
    "congestion_surcharge": congestion_surcharge,
    "Airport_fee": Airport_fee,
    "cbd_congestion_fee": cbd,
})

df.to_parquet("datos/yellow_tripdata_2026-01_sintetico.parquet", index=False)
print(f"Dataset generado: {len(df):,} filas x {df.shape[1]} columnas")
print(f"Nulos en payment_type==0: {(payment_type==0).sum():,} ({(payment_type==0).mean()*100:.1f}%)")
print(df["cbd_congestion_fee"].value_counts())

# --- Tabla de zonas (taxi zone lookup) ---
boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
zone_df = pd.DataFrame({
    "LocationID": zones,
    "Borough": rng.choice(boroughs, size=len(zones), p=[0.4, 0.2, 0.2, 0.12, 0.07, 0.01]),
    "Zone": [f"Zone_{i}" for i in zones],
    "service_zone": rng.choice(["Yellow Zone", "Boro Zone", "Airports", "EWR"], size=len(zones)),
})
zone_df.to_csv("datos/taxi_zone_lookup.csv", index=False)
print(f"\nTabla de zonas: {len(zone_df)} zonas")


### 2. Construcción del esquema estrella, materialización y validación

In [ ]:
"""
SI3008 - Grupo 4  -  ROL A (Modelador/a de datos)
Script 1: Del modelo entidad-relacion al esquema estrella.

Construye, a partir del dataset de taxis:
  - dim_tiempo   (granularidad: hora, NO por marca de tiempo exacta)
  - dim_zona     (desde taxi_zone_lookup)
  - dim_pago     (catalogo de payment_type)
  - dim_tarifa   (catalogo de RatecodeID)
  - fct_viajes   (tabla de hechos, grano = 1 viaje)
Materializa todo en Parquet y genera una version PARTICIONADA por dia.
Valida la integridad referencial.
"""
import os
import pandas as pd
import numpy as np

os.makedirs("lago/bronze", exist_ok=True)
os.makedirs("lago/star", exist_ok=True)
os.makedirs("lago/star/fct_viajes_parquet", exist_ok=True)
os.makedirs("lago/star/fct_viajes_particionado", exist_ok=True)
os.makedirs("lago/csv", exist_ok=True)

# ---------------------------------------------------------------
# 1. Cargar datos
# ---------------------------------------------------------------
# Usa el archivo oficial si existe; si no, el sintetico.
RUTA_OFICIAL = "datos/yellow_tripdata_2026-01.parquet"
RUTA_SINT = "datos/yellow_tripdata_2026-01_sintetico.parquet"
ruta = RUTA_OFICIAL if os.path.exists(RUTA_OFICIAL) else RUTA_SINT
print(f"Cargando: {ruta}")
df = pd.read_parquet(ruta)
# Tabla de zonas: prioriza la OFICIAL de la TLC (265 zonas); si no, la local.
RUTA_ZONAS_OF = "datos/taxi_zone_lookup_oficial.csv"
RUTA_ZONAS = RUTA_ZONAS_OF if os.path.exists(RUTA_ZONAS_OF) else "datos/taxi_zone_lookup.csv"
zonas = pd.read_csv(RUTA_ZONAS)
print(f"  {len(df):,} viajes | {df.shape[1]} columnas | zonas: {RUTA_ZONAS} ({len(zonas)})")

# ---------------------------------------------------------------
# 2. Limpieza minima (consistencia basica antes de modelar)
#    - dropoff >= pickup
#    - solo viajes dentro de enero 2026
# ---------------------------------------------------------------
df = df[df["tpep_dropoff_datetime"] >= df["tpep_pickup_datetime"]]
mes = (df["tpep_pickup_datetime"] >= "2026-01-01") & (df["tpep_pickup_datetime"] < "2026-02-01")
df = df[mes].reset_index(drop=True)
print(f"  Tras limpieza temporal: {len(df):,} viajes")

# ===============================================================
# PASO 1 del esquema estrella: GRANO DEL HECHO
#   Grano = un (1) viaje de taxi.  -> 1 fila en fct_viajes por viaje.
# ===============================================================

# ===============================================================
# PASO 2: DIMENSIONES   (tiempo, zona, pago, tarifa)
# PASO 3: GRANULARIDAD de cada dimension
# PASO 4: CLAVES FORANEAS
# ===============================================================

# ---- dim_tiempo (granularidad = HORA) -------------------------
# Justificacion: el analisis de demanda se hace por hora/dia/dow.
# No se crea una fila por timestamp exacto porque multiplicaria el
# tamano de la dimension sin aportar valor analitico.
horas = pd.date_range("2026-01-01", "2026-02-01", freq="h", inclusive="left")
dim_tiempo = pd.DataFrame({"hora_ts": horas})
dim_tiempo["tiempo_id"] = (dim_tiempo["hora_ts"].dt.strftime("%Y%m%d%H")).astype("int64")
dim_tiempo["fecha"] = dim_tiempo["hora_ts"].dt.date.astype("string")
dim_tiempo["anio"] = dim_tiempo["hora_ts"].dt.year.astype("int16")
dim_tiempo["mes"] = dim_tiempo["hora_ts"].dt.month.astype("int8")
dim_tiempo["dia"] = dim_tiempo["hora_ts"].dt.day.astype("int8")
dim_tiempo["hora"] = dim_tiempo["hora_ts"].dt.hour.astype("int8")
dim_tiempo["dia_semana"] = dim_tiempo["hora_ts"].dt.dayofweek.astype("int8")  # 0=lunes
dim_tiempo["nombre_dia"] = dim_tiempo["hora_ts"].dt.day_name()
dim_tiempo["es_fin_semana"] = dim_tiempo["dia_semana"].isin([5, 6])
dim_tiempo["franja"] = pd.cut(dim_tiempo["hora"],
                              bins=[-1, 5, 11, 17, 21, 23],
                              labels=["Madrugada", "Mañana", "Tarde", "Noche", "Noche"],
                              ordered=False).astype("string")

# ---- dim_zona -------------------------------------------------
dim_zona = zonas.rename(columns={"LocationID": "zona_id",
                                 "Borough": "distrito",
                                 "Zone": "zona",
                                 "service_zone": "zona_servicio"}).copy()
dim_zona["zona_id"] = dim_zona["zona_id"].astype("int64")

# ---- dim_pago -------------------------------------------------
dim_pago = pd.DataFrame({
    "pago_id": [0, 1, 2, 3, 4, 5, 6],
    "metodo_pago": ["Desconocido/No reportado", "Tarjeta de crédito", "Efectivo",
                    "Sin cargo", "Disputa", "Desconocido", "Viaje anulado"],
})

# ---- dim_tarifa (RatecodeID) ----------------------------------
dim_tarifa = pd.DataFrame({
    "tarifa_id": [1, 2, 3, 4, 5, 6, 99],
    "tipo_tarifa": ["Tarifa estándar", "JFK", "Newark", "Nassau/Westchester",
                    "Tarifa negociada", "Viaje compartido", "Nulo/Desconocido"],
})

# ===============================================================
# TABLA DE HECHOS: fct_viajes (grano = 1 viaje)
# ===============================================================
fct = pd.DataFrame()
# claves foraneas
fct["tiempo_id"] = (df["tpep_pickup_datetime"].dt.strftime("%Y%m%d%H")).astype("int64")
fct["zona_pu_id"] = df["PULocationID"].astype("int64")
fct["zona_do_id"] = df["DOLocationID"].astype("int64")
fct["pago_id"] = df["payment_type"].fillna(0).astype("int64")
fct["tarifa_id"] = df["RatecodeID"].fillna(99).astype("int64")
# medidas (hechos aditivos)
fct["trip_distance"] = df["trip_distance"].astype("float32")
fct["fare_amount"] = df["fare_amount"].astype("float32")
fct["tip_amount"] = df["tip_amount"].astype("float32")
fct["tolls_amount"] = df["tolls_amount"].astype("float32")
fct["total_amount"] = df["total_amount"].astype("float32")
fct["cbd_congestion_fee"] = df["cbd_congestion_fee"].astype("float32")
fct["passenger_count"] = df["passenger_count"].astype("float32")
# para particionar
fct["fecha"] = df["tpep_pickup_datetime"].dt.date.astype("string")

# ===============================================================
# VALIDACION DE INTEGRIDAD REFERENCIAL
# ===============================================================
def validar(fk_col, dim_col, dim_df, nombre):
    faltantes = set(fct[fk_col].unique()) - set(dim_df[dim_col].unique())
    ok = len(faltantes) == 0
    print(f"  [{'OK ' if ok else 'FALLA'}] {nombre}: "
          f"{'integridad referencial correcta' if ok else f'claves huerfanas: {faltantes}'}")
    return ok

print("\nValidacion de integridad referencial:")
validar("tiempo_id", "tiempo_id", dim_tiempo, "fct.tiempo_id -> dim_tiempo")
validar("zona_pu_id", "zona_id", dim_zona, "fct.zona_pu_id -> dim_zona")
validar("zona_do_id", "zona_id", dim_zona, "fct.zona_do_id -> dim_zona")
validar("pago_id", "pago_id", dim_pago, "fct.pago_id -> dim_pago")
validar("tarifa_id", "tarifa_id", dim_tarifa, "fct.tarifa_id -> dim_tarifa")

# ===============================================================
# MATERIALIZACION EN PARQUET  (3 configuraciones para el Rol B)
# ===============================================================
# (a) dimensiones
dim_tiempo.drop(columns=["hora_ts"]).to_parquet("lago/star/dim_tiempo.parquet", index=False)
dim_zona.to_parquet("lago/star/dim_zona.parquet", index=False)
dim_pago.to_parquet("lago/star/dim_pago.parquet", index=False)
dim_tarifa.to_parquet("lago/star/dim_tarifa.parquet", index=False)

# (b) hechos - Parquet SIN particion
fct.drop(columns=["fecha"]).to_parquet(
    "lago/star/fct_viajes_parquet/fct_viajes.parquet", index=False)

# (c) hechos - Parquet PARTICIONADO por dia
fct.to_parquet("lago/star/fct_viajes_particionado",
               partition_cols=["fecha"], index=False)

# (d) hechos - CSV (para comparar costo de escaneo en Athena)
fct.drop(columns=["fecha"]).to_csv("lago/csv/fct_viajes.csv", index=False)

print("\nTablas materializadas en lago/star/ (Parquet) y lago/csv/ (CSV).")
print("Particiones creadas (por dia):",
      len([d for d in os.listdir('lago/star/fct_viajes_particionado') if d.startswith('fecha=')]))

# ---------------------------------------------------------------
# Resumen de tamanos (alimenta la tabla del informe)
# ---------------------------------------------------------------
def tam(path):
    if os.path.isfile(path):
        return os.path.getsize(path)
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

print("\nTamanos en disco:")
for etq, p in [("fct CSV", "lago/csv/fct_viajes.csv"),
               ("fct Parquet (sin particion)", "lago/star/fct_viajes_parquet"),
               ("fct Parquet (particionado)", "lago/star/fct_viajes_particionado")]:
    print(f"  {etq:32s}: {tam(p)/1e6:8.2f} MB")

# Guardar metricas para el informe / graficos
metr = {
    "n_viajes": len(fct),
    "n_dim_tiempo": len(dim_tiempo),
    "n_dim_zona": len(dim_zona),
    "csv_mb": tam("lago/csv/fct_viajes.csv")/1e6,
    "parquet_mb": tam("lago/star/fct_viajes_parquet")/1e6,
    "parquet_part_mb": tam("lago/star/fct_viajes_particionado")/1e6,
}
pd.Series(metr).to_json("lago/metricas.json")
print("\nMetricas guardadas en lago/metricas.json")


### 3. Diagramas y gráficos

In [ ]:
"""
SI3008 - Grupo 4 - ROL A
Script 2: Genera los diagramas y graficos para el informe:
  - diagrama_er.png        (modelo entidad-relacion)
  - diagrama_estrella.png  (esquema estrella)
  - fig_tamanos.png        (comparacion de tamanos CSV/Parquet/particionado)
"""
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import pandas as pd

AZUL = "#1f3b73"
AZULC = "#d6e0f5"
VERDE = "#2e7d57"
VERDEC = "#d6efe2"
GRIS = "#555555"

def caja(ax, x, y, w, h, titulo, lineas, fc, ec):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=fc, ec=ec, lw=1.8, zorder=2))
    ax.text(x + w/2, y + h - 0.28, titulo, ha="center", va="top",
            fontsize=10.5, fontweight="bold", color=ec, zorder=3)
    for i, ln in enumerate(lineas):
        ax.text(x + 0.12, y + h - 0.62 - i*0.3, ln, ha="left", va="top",
                fontsize=8.0, color="#222", zorder=3)

# ============================================================
# 1. DIAGRAMA ENTIDAD-RELACION (modelo operacional / OLTP)
# ============================================================
fig, ax = plt.subplots(figsize=(9.5, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 8); ax.axis("off")
ax.set_title("Figura 1. Modelo entidad-relación de la operación de taxis (OLTP)",
             fontsize=11, fontweight="bold", color=AZUL, pad=10)

caja(ax, 4.5, 5.6, 3.0, 2.1, "VIAJE",
     ["PK id_viaje", "pickup_datetime", "dropoff_datetime", "trip_distance",
      "fare_amount, tip, total", "FK id_vendor", "FK PULocationID / DOLocationID",
      "FK payment_type / RatecodeID"], "#fff4d6", "#b8860b")
caja(ax, 0.4, 2.2, 2.6, 1.6, "ZONA", ["PK LocationID", "Borough", "Zone", "service_zone"], AZULC, AZUL)
caja(ax, 3.4, 2.2, 2.4, 1.6, "VENDOR", ["PK VendorID", "nombre"], AZULC, AZUL)
caja(ax, 6.2, 2.2, 2.4, 1.6, "PAGO", ["PK payment_type", "metodo"], AZULC, AZUL)
caja(ax, 9.0, 2.2, 2.6, 1.6, "TARIFA", ["PK RatecodeID", "tipo_tarifa"], AZULC, AZUL)

for (x, y) in [(1.7, 3.8), (4.6, 3.8), (7.4, 3.8), (10.3, 3.8)]:
    ax.add_patch(FancyArrowPatch((6.0, 5.6), (x, y), arrowstyle="-|>",
                                 mutation_scale=14, color=GRIS, lw=1.3,
                                 connectionstyle="arc3,rad=0.0", zorder=1))
ax.text(6.0, 4.75, "N : 1", fontsize=8, color=GRIS, ha="center", style="italic")
plt.tight_layout()
plt.savefig("informe/diagrama_er.png", dpi=150, bbox_inches="tight")
plt.close()

# ============================================================
# 2. DIAGRAMA ESQUEMA ESTRELLA
# ============================================================
fig, ax = plt.subplots(figsize=(9.5, 7))
ax.set_xlim(0, 12); ax.set_ylim(0, 9); ax.axis("off")
ax.set_title("Figura 2. Esquema estrella — grano del hecho: un (1) viaje de taxi",
             fontsize=11, fontweight="bold", color=VERDE, pad=10)

caja(ax, 4.0, 3.2, 4.0, 2.9, "FCT_VIAJES (hechos)",
     ["grano: 1 viaje", "FK tiempo_id", "FK zona_pu_id / zona_do_id",
      "FK pago_id / tarifa_id", "· trip_distance", "· fare / tip / total",
      "· cbd_congestion_fee"], VERDEC, VERDE)

caja(ax, 4.4, 6.6, 3.2, 1.8, "DIM_TIEMPO",
     ["PK tiempo_id (hora)", "fecha, anio, mes, dia", "hora, dia_semana", "franja, fin_semana"], AZULC, AZUL)
caja(ax, 0.3, 4.0, 3.0, 1.7, "DIM_ZONA",
     ["PK zona_id", "distrito (borough)", "zona", "zona_servicio"], AZULC, AZUL)
caja(ax, 8.7, 4.0, 3.0, 1.7, "DIM_PAGO",
     ["PK pago_id", "metodo_pago"], AZULC, AZUL)
caja(ax, 4.4, 0.6, 3.2, 1.7, "DIM_TARIFA",
     ["PK tarifa_id", "tipo_tarifa"], AZULC, AZUL)

cx, cy = 6.0, 4.65
for (x, y) in [(6.0, 6.6), (4.0, 4.85), (8.0, 4.85), (6.0, 2.3)]:
    ax.add_patch(FancyArrowPatch((cx, cy), (x, y), arrowstyle="-|>",
                                 mutation_scale=13, color=GRIS, lw=1.3, zorder=1))
plt.tight_layout()
plt.savefig("informe/diagrama_estrella.png", dpi=150, bbox_inches="tight")
plt.close()

# ============================================================
# 3. GRAFICO DE TAMANOS (CSV vs Parquet vs particionado)
# ============================================================
m = json.load(open("lago/metricas.json"))
fig, ax = plt.subplots(figsize=(7.5, 4.2))
etiquetas = ["CSV", "Parquet\n(sin partición)", "Parquet\n(particionado)"]
valores = [m["csv_mb"], m["parquet_mb"], m["parquet_part_mb"]]
colores = ["#c0504d", VERDE, AZUL]
barras = ax.bar(etiquetas, valores, color=colores, width=0.55)
ax.set_ylabel("Tamaño en disco (MB)")
ax.set_title("Figura 3. Tamaño de la tabla de hechos por formato de almacenamiento",
             fontsize=10.5, fontweight="bold", color=AZUL)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width()/2, v + max(valores)*0.02,
            f"{v:.1f} MB", ha="center", fontsize=9, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("informe/fig_tamanos.png", dpi=150, bbox_inches="tight")
plt.close()

print("Diagramas y graficos generados en informe/")


### 4. Verificación rápida de las tablas resultantes

In [ ]:
import pandas as pd
print('dim_tiempo:'); display(pd.read_parquet('lago/star/dim_tiempo.parquet').head())
print('dim_zona:');   display(pd.read_parquet('lago/star/dim_zona.parquet').head())
print('fct_viajes:'); display(pd.read_parquet('lago/star/fct_viajes_parquet/fct_viajes.parquet').head())